# Capítulo 7: Construcción de aplicaciones de chat
## Inicio rápido de Microsoft Foundry Models API

Este cuaderno está adaptado del [Repositorio de ejemplos de Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) que incluye cuadernos que acceden a los servicios [Azure OpenAI](notebook-azure-openai.ipynb).

> **Nota:** GitHub Models se retirará a finales de julio de 2026. Este cuaderno ahora apunta a [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), que ofrece el mismo catálogo de modelos gratuitos para probar y la experiencia del SDK de Inferencia de Azure AI.


# Resumen  
"Los grandes modelos de lenguaje son funciones que mapean texto a texto. Dado un texto de entrada, un gran modelo de lenguaje intenta predecir el texto que seguirá"(1). Este cuaderno de "inicio rápido" introducirá a los usuarios a conceptos generales de LLM, requisitos principales de paquetes para comenzar con AML, una introducción suave al diseño de prompts, y varios ejemplos breves de diferentes casos de uso. 


## Tabla de Contenidos  

[Descripción general](#overview)  
[Cómo usar el Servicio OpenAI](#how-to-use-openai-service)  
[1. Creando tu Servicio OpenAI](#1.-creating-your-openai-service)  
[2. Instalación](#2.-installation)    
[3. Credenciales](#3.-credentials)  

[Casos de Uso](#use-cases)    
[1. Resumir Texto](#1.-summarize-text)  
[2. Clasificar Texto](#2.-classify-text)  
[3. Generar Nuevos Nombres de Producto](#3.-generate-new-product-names)  
[4. Ajustar un Clasificador](#4.fine-tune-a-classifier)  

[Referencias](#references)


### Construye tu primer prompt  
Este breve ejercicio proporcionará una introducción básica para enviar prompts a un modelo en Microsoft Foundry Models para una tarea simple de "resumen".  


**Pasos**:  
1. Instala la biblioteca `azure-ai-inference` en tu entorno de Python, si aún no lo has hecho.  
2. Carga las bibliotecas auxiliares estándar y configura tus credenciales de Microsoft Foundry Models.  
3. Elige un modelo para tu tarea  
4. Crea un prompt simple para el modelo  
5. ¡Envía tu solicitud a la API del modelo!  


### 1. Instalar `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importar bibliotecas auxiliares e instanciar credenciales


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Encontrar el modelo adecuado  
Modelos como GPT-4o y GPT-4o mini pueden entender y generar lenguaje natural, y están disponibles en el catálogo de Microsoft Foundry Models junto con modelos de Meta, Mistral, Cohere y Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Diseño de Prompts  

"La magia de los grandes modelos de lenguaje es que al ser entrenados para minimizar este error de predicción sobre vastas cantidades de texto, los modelos terminan aprendiendo conceptos útiles para estas predicciones. Por ejemplo, aprenden conceptos como"(1):

* cómo deletrear
* cómo funciona la gramática
* cómo parafrasear
* cómo responder preguntas
* cómo mantener una conversación
* cómo escribir en muchos idiomas
* cómo programar
* etc.

#### Cómo controlar un gran modelo de lenguaje  
"De todas las entradas a un gran modelo de lenguaje, por mucho la más influyente es el prompt de texto(1).

Se puede solicitar a grandes modelos de lenguaje que produzcan salida de varias maneras:

Instrucción: Indica al modelo lo que quieres
Compleción: Induce al modelo a completar el comienzo de lo que quieres
Demostración: Muestra al modelo lo que quieres, ya sea con:
Algunos ejemplos en el prompt
Cientos o miles de ejemplos en un conjunto de datos de entrenamiento para afinamiento fino"



#### Hay tres pautas básicas para crear prompts:

**Mostrar y decir**. Deja claro lo que quieres ya sea mediante instrucciones, ejemplos o una combinación de ambos. Si quieres que el modelo ordene una lista de elementos en orden alfabético o que clasifique un párrafo por sentimiento, muéstrale que eso es lo que quieres.

**Proporciona datos de calidad**. Si estás intentando construir un clasificador o lograr que el modelo siga un patrón, asegúrate de que haya suficientes ejemplos. Asegúrate de corregir tus ejemplos — el modelo generalmente es lo suficientemente inteligente para detectar errores ortográficos básicos y darte una respuesta, pero también podría suponer que esto es intencional y eso puede afectar la respuesta.

**Revisa tus configuraciones.** Las configuraciones temperature y top_p controlan cuán determinista es el modelo al generar una respuesta. Si le pides una respuesta donde solo hay una respuesta correcta, entonces querrás configurar estos parámetros más bajos. Si buscas respuestas más diversas, entonces podrías configurarlos más altos. El error número uno que la gente comete con estas configuraciones es asumir que son controles de "ingenio" o "creatividad".


Fuente: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. ¡Enviar!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Repite la misma llamada, ¿cómo se comparan los resultados?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Resumir texto  
#### Desafío  
Resume el texto añadiendo un 'tl;dr:' al final de un pasaje de texto. Observa cómo el modelo entiende cómo realizar una serie de tareas sin instrucciones adicionales. Puedes experimentar con indicaciones más descriptivas que tl;dr para modificar el comportamiento del modelo y personalizar el resumen que recibes(3).  

Trabajos recientes han demostrado ganancias sustanciales en muchas tareas y puntos de referencia de PLN mediante un preentrenamiento en un gran corpus de texto seguido de un ajuste fino en una tarea específica. Aunque típicamente es una arquitectura agnóstica a la tarea, este método aún requiere conjuntos de datos de ajuste fino específicos para la tarea de miles o decenas de miles de ejemplos. En contraste, los humanos generalmente pueden realizar una nueva tarea de lenguaje con solo unos pocos ejemplos o con instrucciones simples, algo con lo que los sistemas de PLN actuales aún luchan en gran medida. Aquí mostramos que escalar los modelos de lenguaje mejora enormemente el rendimiento agnóstico a la tarea con pocos ejemplos, a veces incluso alcanzando la competitividad con enfoques previos de ajuste fino de última generación. 



Tl;dr


# Ejercicios para varios casos de uso  
1. Resumir texto  
2. Clasificar texto  
3. Generar nuevos nombres de productos


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Clasificar texto  
#### Desafío  
Clasificar elementos en categorías proporcionadas en el momento de la inferencia. En el siguiente ejemplo, proporcionamos tanto las categorías como el texto a clasificar en el aviso (*playground_reference). 

Consulta del cliente: Hola, una de las teclas del teclado de mi portátil se rompió recientemente y necesitaré un reemplazo:

Categoría clasificada:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generar Nuevos Nombres de Productos
#### Desafío
Crea nombres de productos a partir de palabras de ejemplo. Aquí incluimos en la solicitud información sobre el producto para el que vamos a generar nombres. También proporcionamos un ejemplo similar para mostrar el patrón que deseamos recibir. Además, hemos establecido un valor alto de temperatura para aumentar la aleatoriedad y obtener respuestas más innovadoras.

Descripción del producto: Un fabricante de batidos para el hogar
Palabras clave: rápido, saludable, compacto.
Nombres de productos: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descripción del producto: Un par de zapatos que pueden ajustarse a cualquier tamaño de pie.
Palabras clave: adaptable, ajustable, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referencias  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Mejores prácticas para ajustar modelos GPT para clasificar texto](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Para Más Ayuda  
[Equipo de Comercialización de OpenAI](AzureOpenAITeam@microsoft.com) 


# Colaboradores
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
